In [1]:
import numpy as np
import h5py
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.distributions import Categorical
import time
from collections import deque
import matplotlib.pyplot as plt
from tqdm import tqdm
import os
from matplotlib import font_manager as fm

# ==================== 0. 字体与环境配置 ====================
def set_chinese_font():
    try:
        font_paths = ['/usr/share/fonts/truetype/wqy/wqy-microhei.ttc',
                      'C:/Windows/Fonts/simhei.ttf',
                      '/System/Library/Fonts/PingFang.ttc']
        for font_path in font_paths:
            if os.path.exists(font_path):
                fm.fontManager.addfont(font_path)
                plt.rcParams['font.family'] = fm.FontProperties(fname=font_path).get_name()
                plt.rcParams['axes.unicode_minus'] = False
                return True
    except: pass
    return False
set_chinese_font()

# ==================== 1. 卫星抗干扰仿真环境 (保持一致) ====================
class SatelliteEnvV3:
    def __init__(self, h5_path):
        print(f"📂 正在预载入数据集: {os.path.basename(h5_path)} ...")
        with h5py.File(h5_path, 'r') as f:
            self.pred_map = torch.FloatTensor(f['Y_horizon'][:]) 
            self.truth_map = torch.FloatTensor(f['Y_horizon'][:])
            self.type_data = torch.FloatTensor(f['gt_type'][:])
            
        self.num_channels = 10
        self.max_steps = len(self.pred_map)
        self.current_step = 0
        self.last_action = 0

    def reset(self):
        self.current_step = 0
        self.last_action = np.random.randint(0, 9)
        return self._get_state()

    def _get_state(self):
        map_feat = self.pred_map[self.current_step].flatten()
        type_feat = torch.tensor([self.type_data[self.current_step] / 4.0])
        return torch.cat([map_feat, type_feat])

    def step(self, action):
        is_collision = self.truth_map[self.current_step, 0, action] > 0.5
        future_risk = torch.mean(self.pred_map[self.current_step, :, action])
        
        if is_collision:
            reward = -100.0
        else:
            reward = 15.0 - (future_risk.item() * 30.0)
            if action == self.last_action:
                reward += 5.0 
            else:
                reward -= 2.0 
            
        self.last_action = action
        self.current_step += 1
        done = self.current_step >= self.max_steps - 1
        next_state = self._get_state() if not done else torch.zeros(101)
        return next_state, reward, done, is_collision

# ==================== 2. PPO Actor-Critic 网络架构 ====================
class ActorCritic(nn.Module):
    def __init__(self, input_dim=101, output_dim=10):
        super(ActorCritic, self).__init__()
        # 公共特征层
        self.base = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.LayerNorm(512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU()
        )
        # Actor: 输出动作概率分布
        self.actor = nn.Sequential(
            nn.Linear(256, output_dim),
            nn.Softmax(dim=-1)
        )
        # Critic: 输出状态价值 V(s)
        self.critic = nn.Linear(256, 1)

    def forward(self, x):
        features = self.base(x)
        return self.actor(features), self.critic(features)

# ==================== 3. PPO 智能体 ====================
class PPOAgent:
    def __init__(self, state_dim=101, action_dim=10):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.policy = ActorCritic(state_dim, action_dim).to(self.device)
        self.optimizer = optim.AdamW(self.policy.parameters(), lr=1e-4)
        self.policy_old = ActorCritic(state_dim, action_dim).to(self.device)
        self.policy_old.load_state_dict(self.policy.state_dict())
        
        self.gamma = 0.98
        self.eps_clip = 0.2  # PPO 裁剪比例
        self.K_epochs = 4    # 每次更新迭代次数
        self.M_steps = 2000  # 累计多少步更新一次
        self.memory = []

    def choose_action(self, state):
        start_time = time.time()
        state = state.to(self.device).unsqueeze(0)
        with torch.no_grad():
            probs, _ = self.policy_old(state)
        
        dist = Categorical(probs)
        action = dist.sample()
        latency = time.time() - start_time
        return action.item(), dist.log_prob(action), latency

    def update(self):
        if len(self.memory) < self.M_steps: return
        
        # 转换 memory 数据
        states = torch.stack([m[0] for m in self.memory]).to(self.device)
        actions = torch.tensor([m[1] for m in self.memory]).to(self.device)
        logprobs = torch.stack([m[2] for m in self.memory]).to(self.device)
        rewards = []
        discounted_reward = 0
        for reward, is_terminal in reversed([(m[3], m[4]) for m in self.memory]):
            if is_terminal: discounted_reward = 0
            discounted_reward = reward + (self.gamma * discounted_reward)
            rewards.insert(0, discounted_reward)
            
        rewards = torch.tensor(rewards, dtype=torch.float32).to(self.device)
        rewards = (rewards - rewards.mean()) / (rewards.std() + 1e-7)

        for _ in range(self.K_epochs):
            probs, state_values = self.policy(states)
            dist = Categorical(probs)
            logprobs_new = dist.log_prob(actions)
            dist_entropy = dist.entropy()
            
            # 计算 Ratio (pi_theta / pi_theta_old)
            ratios = torch.exp(logprobs_new - logprobs.detach())
            
            # 计算 Surrogate Loss
            advantages = rewards - state_values.detach().squeeze()
            surr1 = ratios * advantages
            surr2 = torch.clamp(ratios, 1-self.eps_clip, 1+self.eps_clip) * advantages
            
            loss = -torch.min(surr1, surr2) + 0.5 * F.mse_loss(state_values.squeeze(), rewards) - 0.01 * dist_entropy
            
            self.optimizer.zero_grad()
            loss.mean().backward()
            self.optimizer.step()
            
        self.policy_old.load_state_dict(self.policy.state_dict())
        self.memory = []

# ==================== 4. 训练执行与指标报告 ====================
def run_ppo_training_with_metrics():
    DATA_PATH = "/root/autodl-tmp/validate/0218/Prediction/sim_dataset_v6_5/academic_long_horizon_v6_5_200k.h5"
    env = SatelliteEnvV3(DATA_PATH)
    agent = PPOAgent()
    
    episodes = 100
    metrics = {'reward': [], 'sr': [], 'hops': [], 'latency': [], 'throughput': [], 'stability': []}
    convergence_ep = -1

    print("🚀 启动 PPO 离线对比训练 (输出格式与 DQN 保持一致)...")
    for ep in range(episodes):
        state = env.reset()
        ep_reward, collisions, steps, hops = 0, 0, 0, 0
        ep_latencies, actions = [], []

        pbar = tqdm(total=env.max_steps, desc=f"Ep {ep+1}/{episodes}", leave=False)
        while True:
            action, log_prob, lat = agent.choose_action(state)
            ep_latencies.append(lat)
            actions.append(action)

            if steps > 0 and action != env.last_action:
                hops += 1
                
            next_state, reward, done, collision = env.step(action)
            agent.memory.append((state, action, log_prob, reward, done))
            
            # PPO 定期批量更新
            if len(agent.memory) >= agent.M_steps:
                agent.update()
                
            state = next_state
            ep_reward += reward
            if collision: collisions += 1
            steps += 1
            pbar.update(1)
            if done: break
        
        pbar.close()
        
        sr = (1 - collisions / steps) * 100
        avg_hops = hops / (steps / 100)
        avg_lat = np.mean(ep_latencies) * 1000
        throughput = (1 - (collisions / steps)) * 1.0
        stability = np.std(actions)

        if convergence_ep == -1 and sr >= 95.0: convergence_ep = ep + 1

        metrics['reward'].append(ep_reward)
        metrics['sr'].append(sr)
        metrics['hops'].append(avg_hops)
        metrics['latency'].append(avg_lat)
        metrics['throughput'].append(throughput)
        metrics['stability'].append(stability)

        print(f"✅ Ep {ep+1} | 成功率: {sr:.2f}% | 每百步跳频: {avg_hops:.1f} | 延迟: {avg_lat:.2f}ms | 吞吐量: {throughput:.2f}")

    print("\n" + "="*50)
    print("📊 PPO 离线实验对比报告总结")
    print("-" * 50)
    print(f"1. 收敛轮次 (Convergence Episode): {convergence_ep if convergence_ep != -1 else '未收敛'}")
    print(f"2. 平均推理时延 (Inference Latency): {np.mean(metrics['latency']):.4f} ms")
    print(f"3. 避障成功率峰值 (Peak Success Rate): {np.max(metrics['sr']):.2f} %")
    print(f"4. 稳态跳频代价 (Avg Switching Cost): {np.mean(metrics['hops'][-10:]):.2f} hops/100steps")
    print(f"5. 归一化吞吐量 (Throughput): {np.mean(metrics['throughput'][-10:]):.4f}")
    print(f"6. 动作稳定性 (Policy Stability Std): {np.mean(metrics['stability'][-10:]):.4f}")
    print("="*50)

    torch.save(agent.policy.state_dict(), "best_ppo_offline_metrics.pth")
    return metrics

if __name__ == "__main__":
    run_ppo_training_with_metrics()

📂 正在预载入数据集: academic_long_horizon_v6_5_200k.h5 ...
🚀 启动 PPO 离线对比训练 (输出格式与 DQN 保持一致)...


✅ Ep 1 | 成功率: 61.20% | 每百步跳频: 89.7 | 延迟: 0.73ms | 吞吐量: 0.61


✅ Ep 2 | 成功率: 62.21% | 每百步跳频: 90.1 | 延迟: 0.69ms | 吞吐量: 0.62


✅ Ep 3 | 成功率: 63.52% | 每百步跳频: 90.6 | 延迟: 0.70ms | 吞吐量: 0.64


✅ Ep 4 | 成功率: 64.12% | 每百步跳频: 89.7 | 延迟: 0.69ms | 吞吐量: 0.64


...

✅ Ep 96 | 成功率: 67.36% | 每百步跳频: 89.8 | 延迟: 0.41ms | 吞吐量: 0.67


✅ Ep 97 | 成功率: 68.54% | 每百步跳频: 88.7 | 延迟: 0.40ms | 吞吐量: 0.69


✅ Ep 98 | 成功率: 68.47% | 每百步跳频: 89.4 | 延迟: 0.43ms | 吞吐量: 0.68


✅ Ep 99 | 成功率: 67.69% | 每百步跳频: 90.2 | 延迟: 0.41ms | 吞吐量: 0.68


✅ Ep 100 | 成功率: 68.95% | 每百步跳频: 89.4 | 延迟: 0.40ms | 吞吐量: 0.69

📊 PPO 离线实验对比报告总结
--------------------------------------------------
1. 收敛轮次 (Convergence Episode): 未收敛
2. 平均推理时延 (Inference Latency): 0.4545 ms
3. 避障成功率峰值 (Peak Success Rate): 68.95 %
4. 稳态跳频代价 (Avg Switching Cost): 89.81 hops/100steps
5. 归一化吞吐量 (Throughput): 0.6763
6. 动作稳定性 (Policy Stability Std): 2.8276
